# MLflow Exercise Background

<img src="images/overview-mlflow-focus.jpg" width=800>

As shown in the figure above, the MLOps platform provides an MLflow service that stores ML model training information to a PostgreSQL database and training artifacts to a MinIO storage service. 

MLflow provides a [Python client](https://mlflow.org/docs/latest/python_api/index.html) for communicating with an MLflow service. For example, we can use the MLflow Python client to start an *MLflow run*, which is an execution of an ML training script:
```python
with mlflow.start_run():
    model = ElasticNet(alpha=..., l1_ratio=...)
    model.fit(train_x, train_y)
```
*MLflow runs* are organized into *MLflow experiments*. An MLflow experiment can be seen as a logical unit of one or more MLflow runs. For example, there can be an MLflow experiment for training an ElasticNet model, and there can be multiple MLflow runs under this experiment for exploring different hyperparameters and/or training datasets.

When starting  an MLflow run, we can record the relevant information, such as the configured hyperparameters and custom evaluation metrics. After the run is completed, we can also upload the produced model artifact to MLflow:
```python
with mlflow.start_run():
    model = ElasticNet(alpha=..., l1_ratio=...)
    model.fit(train_x, train_y)
    mlflow.log_param("alpha", ...)
    mlflow.log_param("l1_ratio", ...)
    mlflow.log_metric("rmse", ...)
    mlflow.sklearn.log_model(model, ...)
```
Below is a complete example. 

More reading material: [MLflow docs](https://mlflow.org/docs/latest/index.html)

# MLflow example
In this example, we'll use sklearn to train a simple ElasticNet model that predicts red wine quality given some chemical attributes. The information of dataset used in this example can be found [here](https://archive.ics.uci.edu/dataset/186/wine+quality). 

### Create an MLflow run
The following code snippet exemplifies how to use the MLflow Python client to record training parameters and evaluation metrics as well as upload the trained model artifact to the MLflow service.

In [1]:
import os
import logging

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNet

# Set an environmental variable named "MLFLOW_S3_ENDPOINT_URL" so that MLflow client knows where to save artifacts.
# The MinIO storage service can be accessed via http://mlflow-minio.local
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://mlflow-minio.local"

# Configure the credentials needed for accessing the MinIO storage service.
# "AWS_ACCESS_KEY_ID" has been configured in a ComfigMap and "AWS_SECRET_ACCESS_KEY" in a Secret in your K8s cluster when you set up the MLOps platform
os.environ["AWS_ACCESS_KEY_ID"] = "minioadmin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "minioadmin"

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

MLFLOW_TRACKING_URI = "http://mlflow-server.local" # This is the URL of the MLflow service
MLFLOW_EXPERIMENT_NAME = "mlflow-minio-test"


def eval_metrics(actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    return rmse


def run_mlflow_example():
    np.random.seed(40)

    # Read the wine-quality csv file from the URL
    csv_url = (
        "http://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
    )

    data = pd.read_csv(csv_url, sep=";")

    # Split the data into training and test sets. (0.75, 0.25) split.
    train, test = train_test_split(data)

    # The predicted column is "quality" which is a scalar from [3, 9]
    train_x = train.drop(["quality"], axis=1)
    test_x = test.drop(["quality"], axis=1)
    train_y = train[["quality"]]
    test_y = test[["quality"]]
    
    # Just use hard-coded hyperparameters
    alpha = 0.5
    l1_ratio = 0.5

    logger.info(f"Using MLflow tracking URI: {MLFLOW_TRACKING_URI}")

    # Configure the MLflow client to connect to the MLflow service
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

    logger.info(f"Using MLflow experiment: {MLFLOW_EXPERIMENT_NAME}")
    mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

    with mlflow.start_run() as run:
        print("MLflow run_id:", run.info.run_id) # Each MLflow Run has a unique identifier called run_id

        lr = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42)

        logger.info("Fitting model...")

        lr.fit(train_x, train_y)

        logger.info("Finished fitting")

        predicted_qualities = lr.predict(test_x)

        rmse = eval_metrics(test_y, predicted_qualities)

        logger.info("Elasticnet model (alpha=%f, l1_ratio=%f):" %
                    (alpha, l1_ratio))
        logger.info("  RMSE: %s" % rmse)


        logger.info("Logging parameters to MLflow")
        mlflow.log_param("alpha", alpha)
        mlflow.log_param("l1_ratio", l1_ratio)
        mlflow.log_metric("rmse", rmse)

        logger.info("Logging trained model")
        artifact_name = "model"
        mlflow.sklearn.log_model(
            lr, artifact_name, registered_model_name="ElasticnetWineModel")
        print("The S3 URI of the logged model:", mlflow.get_artifact_uri(artifact_path=artifact_name))

In [2]:
run_mlflow_example()

INFO:__main__:Using MLflow tracking URI: http://mlflow-server.local
INFO:__main__:Using MLflow experiment: mlflow-minio-test
INFO:__main__:Fitting model...
INFO:__main__:Finished fitting
INFO:__main__:Elasticnet model (alpha=0.500000, l1_ratio=0.500000):
INFO:__main__:  RMSE: 0.793164022927685
INFO:__main__:Logging parameters to MLflow
INFO:__main__:Logging trained model


MLflow run_id: 565a221aa54348929ca8da07edf1973a


INFO:botocore.credentials:Found credentials in environment variables.
Registered model 'ElasticnetWineModel' already exists. Creating a new version of this model...
2025/11/26 20:06:06 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticnetWineModel, version 9


The S3 URI of the logged model: s3://mlflow/4/565a221aa54348929ca8da07edf1973a/artifacts/model


Created version '9' of model 'ElasticnetWineModel'.


Expected output:

```text
INFO:__main__:Using MLflow tracking URI: http://mlflow-server.local
INFO:__main__:Using MLflow experiment: mlflow-minio-test
2023/08/16 13:45:22 INFO mlflow.tracking.fluent: Experiment with name 'mlflow-minio-test' does not exist. Creating a new experiment.
(More logs...)

MLflow run_id: fca4fffeab3d44e98ad2584f9f32a45
Successfully registered model 'ElasticnetWineModel'.
The S3 URI of the logged model: s3://mlflow/7/fca4fffeab3d44e98ad2584f9f32a45a/artifacts/model
```
Note that the S3 URI of the logged model vary. 

Navigate to the MLflow service UI at [http://localhost:5000/](http://localhost:5000/) (ported from the http://mlflow-server.local URL inside the cluster),
and you should see your run under the experiment "mlflow-minio-test". You can browse the run parameters, metrics and artifacts. For example: 

* Training hyperparameters and evaluation metrics:

<img src="images/mlflow-logging.png" width="1000"/>

You may notice that the "Metrics" and "Parameters" field are hidden by default, you can make them visible by clicking the "Columns" tab:

<img src="images/mlflow-show-columns.png" width=1000 />

When clicking the Run Name, we can also check where the model and other related files have been uploaded:

<img src="images/mlflow-uploaded-artifacts.png" width=1000 />

In this case, the model (which is a Pickle file) and its related files (such as the model dependency requirements) have been uploaded to the MinIO service. Navigate to [http://localhost:9001/](http://localhost:9001/) (ported from the http://mlflow-minio.local URL inside the cluster) and login using "minioadmin" as both the username and password, we can see there is a bucket named "mlflow":

<img src="images/minio-bucket-ui.png" width=1000 />

clicking the bucket (and its underlying folders) we can see the model and its related artifacts reside in the "mlflow" bucket:

<img src="images/minio-model-artifacts.png" width=1000 />

* Finally, we can also see the model has been registered to MLflow:

<img src="images/mlflow-model.png" width="1000"/>

# MLflow exercise
**Please do the exercise using the `mlops_eng` environment.**

In this exercise, you will train a LightGBM regression model to predict public bike sharing demand given attributes like datetime and weather conditions. The dataset ("bike_sharing_demand.csv", located under the same directory as this notebook) used in this exercise is a preprocessed version of [this kaggle dataset](https://www.kaggle.com/competitions/bike-sharing-demand/overview). You'll learn more about data preprocessing next week. Additionally, you'll use MLflow to track the model training and Deepchecks to evaluate the trained model. 

In [3]:
import pandas as pd
from lightgbm import LGBMRegressor
import mlflow
from deepchecks.tabular import Suite
from deepchecks import SuiteResult, CheckResult
from deepchecks.tabular.checks import TrainTestPerformance, ModelInferenceTime, MultiModelPerformanceReport
from deepchecks.tabular import Dataset
from pathlib import Path
import os
import pickle
import boto3
import logging
import warnings

from typing import Tuple, Dict, Any

PARENT_DIR = Path("").parent.absolute()

# Suppress boto3 logging
boto3.set_stream_logger(name='botocore.credentials', level=logging.ERROR)

In [4]:
def delete_file_if_existing(filename: str):
    """
    Delete a file if it exists
    """
    if os.path.exists(filename):
        print(f"Delete the existing {filename}")
        os.remove(filename)


## Task 1: Download and split the dataset
Please note that the dataset (bike_sharing_demand.csv) contains data collected from two years so you'll find duplicated data points for most of the timestamps (hour-day-month) if you explore the dataset. 

### 1a) Load the data
First, implement the `pull_data` function that loads a CSV as a Pandas DataFrame from a given location.

In [5]:
def pull_data(dataset_path: Path) -> pd.DataFrame:
    """
    Download the data set from a given path
    Args: 
        dataset_path (Path): Path of the CSV 
    Returns:
        A Pandas DataFrame of the dataset
    """
    ### START CODE HERE
    data = pd.read_csv(dataset_path)
    return data
    ### END CODE HERE


In [6]:
# You can use this code cell to check if your pull_data function works correctly
dataset_path = PARENT_DIR / "bike_sharing_demand.csv"
df = pull_data(dataset_path)
assert df.shape == (10886, 12)


In [7]:
# Show a concise summary of the DataFrame
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10886 entries, 0 to 10885
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   season      10886 non-null  int64  
 1   holiday     10886 non-null  int64  
 2   workingday  10886 non-null  int64  
 3   weather     10886 non-null  int64  
 4   temp        10886 non-null  float64
 5   atemp       10886 non-null  float64
 6   humidity    10886 non-null  int64  
 7   windspeed   10886 non-null  float64
 8   count       10886 non-null  int64  
 9   hour        10886 non-null  int64  
 10  day         10886 non-null  int64  
 11  month       10886 non-null  int64  
dtypes: float64(3), int64(9)
memory usage: 1020.7 KB


<details>
    <summary>Expected output</summary>
    <img src="./images/dataset-info.png"/>
</details>

Below is the explanation of each column in the dataset:

**Variables**:

| Column name |  Explanation | type |
|-------------|---------------|----|
| season      | 1 = spring, 2 = summer, 3 = fall, 4 = winter | integer
| holiday     | whether the day is considered a holiday | integer
| workingday  | 1 if day is neither weekend nor holiday, otherwise 0. | integer
| weather     | 1: Clear, Few clouds, Partly cloudy, Partly cloudy; 2: Mist + Cloudy, Mist + Broken clouds, Mist + Few clouds, Mist; 3: Light Snow, Light Rain + Thunderstorm + Scattered clouds, Light Rain + Scattered clouds; 4: Heavy Rain + Ice Pallets + Thunderstorm + Mist, Snow + Fog | integer
| temp        | temperature in Celsius | float
| atemp       | "feels like" temperature in Celsius | float
| humidity    | relative humidity | integer
| windspeed   | wind speed | float
| hour        | the hours of the datetime| integer
| day         | the day of the datetime| integer
| month       | the month of the datetime| integer

**Targets**: 

| Column name | Explanation                                     | Type
|-------------|-------------------------------------------------| ----|
| count       | number of total rentals                         | integer

### 1b) Split the data into train and test DataFrames
Then implement the `splitData` function that splits the dataset into a training and a test dataset, using the last 168 rows of the dataset as the test data.

In [12]:
def splitData(input_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split a DataFrame into training and testing sets
    Args:
        input_df (DataFrame): The DataFrame to be split
    Returns:
        A tuple of training and testing DataFrame
    """
    ### START CODE HERE
    train_df = input_df[:-168]
    test_df = input_df[-168:]
    return train_df, test_df
    ### END CODE HERE

In [13]:
df = pull_data(dataset_path)
train, test = splitData(df)

In [15]:
# Check if train and test DataFrames are split correctly
expected_train_shape = (10718, 12)
expected_test_shape = (168, 12)

assert (
    train.shape == expected_train_shape
), "The dimension of the training dataset is not correct"
assert (
    test.shape == expected_test_shape
), "The dimension of the testing dataset is not correct"

expected_columns = [
    "season",
    "holiday",
    "workingday",
    "weather",
    "temp",
    "atemp",
    "humidity",
    "windspeed",
    "count",
    "hour",
    "day",
    "month",
]
assert set(train.columns) == set(
    expected_columns
), "The columns of the training dataset are not correct"
assert set(test.columns) == set(
    expected_columns
), "The columns of the training dataset are not correct"

In [16]:
# Split the training and testing DataFrames into features and targets
target = "count"
input_df = pull_data(PARENT_DIR / "bike_sharing_demand.csv")
train, test = splitData(input_df)
train_x = train.drop([target], axis=1)
test_x = test.drop([target], axis=1)
train_y = train[[target]]
test_y = test[[target]]

## Task 2: Offline model evaluation using Deepchecks

### 2a) Construct the Dataset objects used by Deepchecks
First, let's construct the Deepchecks Dataset objects (named `train_dataset` and `test_dataset`) from the train and testing DataFrames.

**Note**: Please use the `categorical_features` variable given below to specify the categorical features when you construct the datasets handled by Deepchecks. Please check [here](https://docs.deepchecks.com/stable/tabular/usage_guides/dataset_object.html) for more details. 

In [18]:
# Categorical features, remember to specify the categorical features when you construct the datasets handled by Deepchecks
# See https://docs.deepchecks.com/stable/tabular/usage_guides/dataset_object.html
categorical_features = ["season", "holiday", "workingday", "weather", "hour", "day", "month"]

# TODO: train_dataset = ...
# test_dataset = ...
### START CODE HERE
train_dataset = Dataset(train_x, label=train_y, cat_features=categorical_features)
test_dataset = Dataset(test_x, label=test_y, cat_features=categorical_features)
### END CODE HERE

In [19]:
# Categorical features should be specified in train_dataset and test_dataset
assert sorted(train_dataset.cat_features) == sorted(categorical_features), "The categorical features of train_dataset are not specified correctly"
assert sorted(test_dataset.cat_features) == sorted(categorical_features),   "The categorical features of test_dataset are not specified correctly"

# train_dataset and test_dataset should have the correct feature and label columns
assert (train_dataset.features_columns.shape) == (10718, 11), "The features columns of train_dataset are not correct"
assert (test_dataset.features_columns.shape) == (168, 11), "The features columns of test_dataset are not correct"
assert (train_dataset.label_name) == "count", "The label name of train_dataset is not correct"
assert (test_dataset.label_name) == "count", "The label name of test_dataset is not correct"

### 2b) Deepchecks Suite with conditions
Implement the `evaluate` function that uses Deepchecks Suite to perform the following two tests:
1) Evaluate the model's MAE and RMSE on both training and testing dataset. This test should fail if the MAE or RMSE drops more than 20% on the testing dataset compared to the training dataset;
2) Evaluate the model's inference time on both training and tests dataset. This test should fail if the average inference time exceeds 0.1 second. 

Finally, this function should return a Deepchecks [SuiteResult](https://docs.deepchecks.com/stable/api/generated/deepchecks.core.SuiteResult.html) containing the evaluation result.

**Hints**:
- [How to add conditions to a test?](https://docs.deepchecks.com/stable/general/usage/customizations/auto_examples/plot_configure_check_conditions.html)
- [Train test performance](https://docs.deepchecks.com/stable/api/generated/deepchecks.tabular.checks.model_evaluation.TrainTestPerformance.html)
- [Model inference time](https://docs.deepchecks.com/stable/tabular/auto_checks/model_evaluation/plot_model_inference_time.html)
- [Condition for comparing model performance between training and testing dataset](https://docs.deepchecks.com/stable/api/generated/deepchecks.tabular.checks.model_evaluation.TrainTestPerformance.add_condition_train_test_relative_degradation_less_than.html#deepchecks.tabular.checks.model_evaluation.TrainTestPerformance.add_condition_train_test_relative_degradation_less_than)
- [Condition for validating inference time](https://docs.deepchecks.com/stable/api/generated/deepchecks.tabular.checks.model_evaluation.ModelInferenceTime.add_condition_inference_time_less_than.html)

In [ ]:
def evaluate(train_dataset: Dataset, test_dataset: Dataset, model: LGBMRegressor) -> SuiteResult:
    """
    Use Deepchecks to evaluate 1) model's MAE and RMSE on both training and testing dataset, 2) model's inference time.
    Args:
        train_dataset (Dataset): training Dataset
        test_dataset (Dataset): testing Dataset
        model (LGBMRegressor): The LightGBM regression model to be evaluated
    Return:
        a Deepchecks SuiteResult that contains the results of a Deepchecks suite run
    """
    
    ### START CODE HERE
    raise NotImplementedError()
    ### END CODE HERE


In [ ]:
# We provide a testing model trained on the same bike demand dataset to help you check if your evaluate function works correctly
test_model = pickle.load(open("test-model.pkl", "rb"))
evaluation_result = evaluate(train_dataset, test_dataset, test_model)

# These tests should pass
failed_checks = evaluation_result.get_not_passed_checks()
assert len(failed_checks) == 1, "The number of failed checks in the evaluation result is not correct"
failed_condition_result = failed_checks[0].conditions_results[0]
assert failed_condition_result.name == "Train-Test scores relative degradation is less than 0.2", "The condition for comparing model performance between the training and test dataset is not correct"

passed_checks = evaluation_result.get_passed_checks()
assert len(passed_checks) == 2, "The number of passed checks in the evaluation result is not correct"
passed_condition_result = passed_checks[0].conditions_results[0]
assert passed_condition_result.name == "Average model inference time for one sample is less than 0.1",  "The condition for evaluating model inference time is not correct"

In [ ]:
# Export the evaluation results to an HTML file
evaluation_result_file = "test-result.html"
delete_file_if_existing(evaluation_result_file)

evaluation_result_file = evaluation_result.save_as_html(evaluation_result_file)
print(f"The evaluation result is saved in an HTML file named {evaluation_result_file}")

After running the above code cell, you should see a file named "test-result.html" appear under the same directory as this notebook.

<details>
    <summary> Expected output when open the file in your browser </summary>
    <br />
    The test of MAE/RMSE should fail:
    <br />
    <img src="images/deepchecks-train-test-performance.png">
    <br />
    The test of inference time should pass:
    <br />
    <img src="images/deepchecks-inference-time.png">
</details>

## Assignment 3: Tracking model training in MLflow
Similar to what you see in the MLflow tutorial, please complete the `log_to_mlflow` function that performs the following tasks:
1. Use LightGBM to train a regression model to predict the bike sharing demand. The model should be trained using the training DataFrame you prepared previously and with the hyperparameters given as an argument. The hyperparameters are given as a dictionary, e.g. `hyperparams = {"num_leaves": 63, "learning_rate": 0.05, "random_state": 42}`. 
1. In an MLflow Run, use MLflow to track the model training:
    1. Log the hyperparameters used to MLflow, using the keys in the `hyperparams` dictionary as the parameter names as shown below (check the "Parameters" column in the screenshot below).
    2. Use the `evaluation` function you created above to evaluate the trained model. Then export the evaluation results to an HTML file and upload the file to MLflow. The HTML file of the Deepchecks model evaluation results uploaded to MLflow should be named **"evaluation_result.html"**. Please also make sure that the file you upload to MLflow is not inside any subdirectory of the MLflow Run.   
    3. Register the trained model to MLflow. The registered model should be named **"Week1LgbmBikeDemand"** and linked to the MLflow Run. 
    4. Finally, return the Run ID of the MLflow Run. (You can refer to the MLflow tutorial on how to get the Run ID of an MLflow Run.)

More illustration:

<img src="./images/ass3-example.png" width=1200/>

Hints:
* [Log multiple parameters to MLflow](https://mlflow.org/docs/2.9.2/python_api/mlflow.html#mlflow.log_params)
* [Log a local file to MLflow](https://mlflow.org/docs/2.9.2/python_api/mlflow.html#mlflow.log_artifact)
* [Log a model to MLflow](https://mlflow.org/docs/2.9.2/python_api/mlflow.lightgbm.html?highlight=log_model#mlflow.lightgbm.log_model)

In [ ]:
# mlflow configuration
MLFLOW_S3_ENDPOINT_URL = "http://mlflow-minio.local"
MLFLOW_TRACKING_URI = "http://mlflow-server.local"
AWS_ACCESS_KEY_ID = "minioadmin"
AWS_SECRET_ACCESS_KEY = "minioadmin"
mlflow_experiment_name = "week1-lgbm-bike-demand"

os.environ["MLFLOW_S3_ENDPOINT_URL"] = MLFLOW_S3_ENDPOINT_URL
os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(mlflow_experiment_name)

In [ ]:
mlflow_evaluation_result_filename = "evaluation_result.html"
registered_model_name = "Week1LgbmBikeDemand"

def log_to_mlflow(hyperparams: Dict[str, Any]) -> str:
    """
    Train a LightGBM model, log the hyperparameters used, upload the Deepchecks evaluation result HTML file, and register the trained model to MLflow
    Args:
        hyperparams: The hyperparameters used to train the model
    Returns:
        The MLflow Run ID
    """
    with mlflow.start_run() as run:
        model = LGBMRegressor(**hyperparams)
        model.fit(train_x, train_y, categorical_feature=categorical_features)

        # TODO: 1) Log hyperparameters
        # 2) Use the "evaluate" function to evaluate the model, export the evaluation results to an HTML file and upload the file
        # 3) Register the model
        # 4) Return the MLflow Run ID
        ### START CODE HERE
        raise NotImplementedError()
        ### END CODE HERE

In [ ]:
# model hyperparameters
hyperparams = {
    "num_leaves": 63,
    "learning_rate": 0.05,
    "random_state": 42
}

delete_file_if_existing(mlflow_evaluation_result_filename)
mlflow_run_id = log_to_mlflow(hyperparams=hyperparams)
print(f"MLflow Run ID: {mlflow_run_id}")

In [ ]:
# This test simply checks that your mlflow_run_id is not empty
# Additional tests will be used during the grading so please make sure your implementation satisfies the requirements listed in the assignment instructions
assert len(mlflow_run_id) != 0


### Screenshots to be submitted for Assignment 3
To get the points from Assignment 3, please submit the following screenshots:
1. The logs of your MLflow run. Please include the parameters of the model in your screenshot. 
<details>
    <summary>Example</summary>
    <img src="images/mlflow-run.png" width=1000>
</details>

2. The details of the MLflow run, including the uploaded Deepchecks evaluation result file;
<details>
    <summary>Example</summary>
    <img src="images/mlflow-run-detail1.png" width=1000>
    <img src="images/mlflow-run-detail2.png" width=1000>
    <img src="images/mlflow-run-detail3.png" width=1000>
</details>

3. The registered model.
<details>
    <summary>Example</summary>
    <img src="images/mlflow-model.png" width=1000>
</details>

## Task 4: Evaluate the trained model against another model
Suppose your colleague had trained an ElasticNet model for the same use case of bike sharing demand prediction. In this assignment, please complete the `compare_models` function that performs the following tasks:
1. Using Deepchecks [Multi model performance report](https://docs.deepchecks.com/stable/tabular/auto_checks/model_evaluation/plot_multi_model_performance_report.html) to compare MAE and RMSE of your LightGBM model to the ElasticNet model and saving the result as an HTML file. **Use negative MAE and negative RMSE as the scorers**.
1. Uploading the result file to MLflow, the file should be named **"model_comparison.html"** and under the MLflow Run where you trained your LightGBM model in Task 3. Similar to `evaluation_result.html`, `model_comparison.html` shouldn't be inside any other folder.
E.g.,

<img src="images/ass4-example.png" width=300 />

`model_comparison.html` should look like this:

<img src="images/deepchecks-compare-models.png" >
Finally, the function should return a Deepckecks [CheckResult](https://docs.deepchecks.com/stable/api/generated/deepchecks.core.CheckResult.html) containing the model comparison results. 

Note that the idea here is to attach a file to an existing MLflow Run, not to create a new MLflow Run and then upload the file under the new MLflow Run. 

You may find the following doc helpful: 
- [mlflow.start_run](https://mlflow.org/docs/2.9.2/python_api/mlflow.html#mlflow.start_run) (Pay attention to the use of the `run_id` parameter).

In [ ]:
# This is the model you just trained. In practice, the model can be downloaded from MLflow. Here, for simplicity, we just retrain the model
model = LGBMRegressor(**hyperparams)
model.fit(train_x, train_y, categorical_feature=categorical_features)

# Load the old ElasticNet model
old_model = pickle.load(open("old-model.pkl", "rb"))

In [ ]:
mlflow_model_comparison_result_filename = "model_comparison.html"

def compare_models(mlflow_run_id: str) -> CheckResult:
    """
    Use Deepchecks to compare the performance of the LightGBM model and the old ElasticNet model
    Args:
        mlflow_run_id: The model comparison result file should be uploaded under the MLflow Run whose Run ID is mlflow_run_id
    Return:
        Deepchecks CheckResult that contains the model comparison results
    """

    # TODO: 1) Use Deepchecks to compare your LightGBM model to the ElasticNet model and save the result to an HTML file
    # 2) Upload the result file to MLflow. The file should be under the MLflow Run where you trained your LightGBM model
    # 3) Return the comparison CheckResult
    ### START CODE HERE
    raise NotImplementedError()
    ### END CODE HERE

In [ ]:
delete_file_if_existing(mlflow_model_comparison_result_filename)

os.environ["MLFLOW_S3_ENDPOINT_URL"] = MLFLOW_S3_ENDPOINT_URL
os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)


In [ ]:
res = compare_models(mlflow_run_id=mlflow_run_id)

# Check that the returned CheckResult is correct
res_df = res.value
lgbm_neg_mae = res_df.loc[
    (res_df["Model"] == "LGBMRegressor") & (res_df["Metric"] == "neg_mae")
]["Value"].values[0]
elasticnet_neg_mae = res_df.loc[
    (res_df["Model"] == "ElasticNet") & (res_df["Metric"] == "neg_mae")
]["Value"].values[0]
assert (
    lgbm_neg_mae > elasticnet_neg_mae
), "The Deepchecks model comparison report is not correct. The negative MAE of the LightGBM model should be larger than the negative MAE of the ElasticNet model"

lgbm_neg_rmse = res_df.loc[
    (res_df["Model"] == "LGBMRegressor") & (res_df["Metric"] == "neg_rmse")
]["Value"].values[0]
elasticnet_neg_rmse = res_df.loc[
    (res_df["Model"] == "ElasticNet") & (res_df["Metric"] == "neg_rmse")
]["Value"].values[0]
assert (
    lgbm_neg_rmse > elasticnet_neg_rmse
), "The Deepchecks model comparison report is not correct. The negative RMSE of the LightGBM model should be larger than the negative RMSE of the ElasticNet model"

delete_file_if_existing(mlflow_model_comparison_result_filename)


### Expected output for Task 4
The details of the MLflow run including uploaded Deepchecks model comparison result file.
<details>
    <summary>Example</summary>
    <img src="images/deepchecks-compare-models.png" />
</details>